In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Decision Tree
Decision Trees are models that performs in regression and classification tasks.It works with decision rules infered in the data features.the more deeper the tree,more complex it rules

In [3]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import export_graphviz

iris = load_iris()
x = iris.data[:,1:3]
y = iris.target


tree_clf = DecisionTreeClassifier(max_depth=2)
tree_clf.fit(x,y)

export_graphviz( tree_clf, out_file="iris_tree.dot", feature_names=iris.feature_names[2:], class_names=iris.target_names, rounded=True, filled=True)


image:

[text](iris_tree.dot)

In [4]:
print(tree_clf.predict_proba([[5,1.4]])) #prob of each class 0,1,2
print(tree_clf.predict([[5,1.4]])) #class

[[1. 0. 0.]]
[0]


here,the model separates each class and use decision rules to select the best class based on ou features 2 and 3.

Regression

In [5]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.datasets import load_diabetes

data = load_diabetes(scaled=True,return_X_y=False)
y = data.target
x = data.data

tree_reg = DecisionTreeRegressor(max_depth=2)
tree_reg.fit(x,y)


DecisionTreeRegressor(max_depth=2)

In [6]:
#data moons
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

x,y = make_moons()
x_tr,x_ts,y_tr,y_ts = train_test_split(x,y,test_size=0.2)

## Voting Classifier
The idea behind this type of model is to combine the predicted classes or class proba mean of mutiples models and select the majority class as a final result.

- Soft Voting : uses classes predict proba mean,which can be regulated by weights in each model.
- Hard Voting : uses models predict classes count

In [7]:
from sklearn.ensemble import RandomForestClassifier,VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

#models:
log_rg = LogisticRegression()
rf = RandomForestClassifier()
svm = SVC()

voting_cls = VotingClassifier(
    estimators=[('lg',log_rg),('svm',svm),('rf',rf)],
    voting='hard'
)

voting_cls.fit(x_tr,y_tr)

VotingClassifier(estimators=[('lg', LogisticRegression()), ('svm', SVC()),
                             ('rf', RandomForestClassifier())])

In [8]:
for model in (log_rg,rf,svm,voting_cls):
    model.fit(x_ts,y_ts)
    y_pred = model.predict(x_ts)
    print(model.__class__.__name__,accuracy_score(y_ts,y_pred))

LogisticRegression 0.85
RandomForestClassifier 1.0
SVC 1.0
VotingClassifier 1.0


### Bagging and Pasting 
In a contrary way,we use the same model but with diferents samples.basically we train mutiples estimators of the same model with diferents samples,being bootstrap (bagging) or without reposition (pasting).

bagging uses bootstrap while pasting don't.

with bagging or pasting algorithm,the variance tends to decreases compared with original model.

In [9]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf = BaggingClassifier(
    DecisionTreeClassifier(),n_estimators=500,
    max_samples=70,bootstrap=True,n_jobs=-1,
    oob_score=True
)

past_clf = BaggingClassifier(
    DecisionTreeClassifier(),n_estimators=100,
    max_samples=70,n_jobs=-1,bootstrap=False,
    
)

bag_clf.fit(x_tr,y_tr)
bag_y_pred = bag_clf.predict(x_ts)

past_clf.fit(x_tr,y_tr)
past_clf_y_pred=past_clf.predict(x_ts)

For bagging,we can also evaluate the _Out of Bag_,which corresponds to the data out of sampling.it happens beacause the bootstrap uses a normal distribuition,where 60%~ of data are in the mean.the out 40%~ are less frequent and the sampling method don't know this data.

In [10]:
##accuracy:
bag_s = accuracy_score(y_ts,bag_y_pred)
past_s = accuracy_score(y_ts,past_clf_y_pred,)


oob_bag = bag_clf.oob_score_
print(f'bag: score= {bag_s},oob= {oob_bag} \n past: score= {past_s}')

bag: score= 0.95,oob= 0.9375 
 past: score= 0.9


### Random Forest
The Bagging that you did see above is a model called _Random Forest_.this model basically uses several decision trees with several samples to find best features between data.all it using nodes 

In [11]:
from sklearn.ensemble import RandomForestClassifier

r_forest = RandomForestClassifier(
    n_estimators=500,max_leaf_nodes=16,n_jobs=-1
)

r_forest.fit(x_tr,y_tr)
rf_pred = r_forest.predict(x_ts)
print(f'rf score : {accuracy_score(y_ts,rf_pred)}')

rf score : 0.95


You can either use the bagging alg;

In [12]:
rf_bag = BaggingClassifier(
    DecisionTreeClassifier(splitter='random',
                    max_leaf_nodes=16),n_estimators=500,
    max_samples=1.0,bootstrap=True,n_jobs=-1,
    )

rf_bag.fit(x_tr,y_tr)
rf_bag_pred = rf_bag.predict(x_ts)
print(f'rf bag score : {accuracy_score(y_ts,rf_bag_pred)}')

rf bag score : 1.0


One important thing about Random Forests is that they meansure the importance coeficient of each feature.the model computes the impurity of any feature on nodes and use the weight of each.

In [13]:
from sklearn.datasets import load_iris
iris = load_iris()

rf = RandomForestClassifier(n_estimators=500,n_jobs=-1)

rf.fit(iris['data'],iris['target'])

RandomForestClassifier(n_estimators=500, n_jobs=-1)

In [14]:
for feature,score in zip(iris['feature_names'],rf.feature_importances_):
    print(f'feature: {feature},score: {score}')

feature: sepal length (cm),score: 0.09035107626914361
feature: sepal width (cm),score: 0.025479709394726602
feature: petal length (cm),score: 0.44656689200463306
feature: petal width (cm),score: 0.4376023223314967


### Boosting
If we would use a voting plus a random forest with bagging or pasting?

The boosting algorithms work using a weak learners to combine them in one.each model computes the misclassified previous instances and put weights based in its impurity to make a prediction.the next model after this computes again using previous weights and its predictions.these models are combined sequentially to create a **AdaBoosting**.

we can use the **learning rate** to put the weights of each predictor,like a gradient

In [26]:
from sklearn.ensemble import AdaBoostClassifier

ada_boost = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1), #n1--n2,n3 ; one decision rule for each tree
    n_estimators=10,algorithm="SAMME",learning_rate=0.5
)

ada_boost.fit(x_tr,y_tr)

e:\Users\PC\Games\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


AdaBoostClassifier(algorithm='SAMME',
                   estimator=DecisionTreeClassifier(max_depth=1),
                   learning_rate=0.5, n_estimators=10)

In [27]:
for errors,estimator,weights in zip(ada_boost.estimator_errors_,ada_boost.estimators_,
                                           ada_boost.estimator_weights_):
    print(f'''estimator: {estimator},errors: {errors},weights: {weights}''')

estimator: DecisionTreeClassifier(max_depth=1, random_state=495873847),errors: 0.15000000000000005,weights: 0.867300527694053
estimator: DecisionTreeClassifier(max_depth=1, random_state=248835023),errors: 0.22928871518601449,weights: 0.6061659302531536
estimator: DecisionTreeClassifier(max_depth=1, random_state=1380421445),errors: 0.19789013155800225,weights: 0.699766802994681
estimator: DecisionTreeClassifier(max_depth=1, random_state=313954841),errors: 0.23749024370732782,weights: 0.5832443800168777
estimator: DecisionTreeClassifier(max_depth=1, random_state=1570712984),errors: 0.19576602800778314,weights: 0.7064850126383754
estimator: DecisionTreeClassifier(max_depth=1, random_state=196520655),errors: 0.2982343758855901,weights: 0.4278599029283324
estimator: DecisionTreeClassifier(max_depth=1, random_state=2055117474),errors: 0.26132152191560987,weights: 0.5195556076741352
estimator: DecisionTreeClassifier(max_depth=1, random_state=1062842357),errors: 0.2980232994295254,weights: 0.4